In [ ]:
import requests
import pandas as pd

In [ ]:
# List of Dow 30 stock tickers
symbols = [
    "AXP", "AMGN", "AAPL", "BA", "CAT", "CSCO", "CVX", "GS", "HD", "HON",
    "IBM", "INTC", "JNJ", "KO", "JPM", "MCD", "MMM", "MRK", "MSFT", "NKE",
    "PG", "TRV", "UNH", "CRM", "VZ", "V", "WBA", "WMT", "DIS", "DOW"
]

In [ ]:
# Initialize an empty list to store DataFrames for each symbol
dfs = {}

for i in range(1, len(symbols)+1):

    url = f'https://eodhd.com/api/eod/{symbols[i-1]}.US?from=2014-05-20&to=2024-05-20&api_token=5ffde981a31603.50022638&fmt=json' # with free API token, I can only do over the past year
    nifty_data = requests.get(url).json()

    #print(data)

    # Convert data to DataFrame
    dfs[f'df{i}'] = pd.DataFrame(nifty_data)

    # changing the date to the index column
    dfs[f'df{i}'] = dfs[f'df{i}'][['date', 'close']].set_index('date')

    # renaming the 'close' column to the name of the symbol
    dfs[f'df{i}'] = dfs[f'df{i}'].rename(columns={'close': symbols[i-1]})

    #print(dfs[f'df{i}'])



# Join the data frames in the dictionary
result_df = pd.DataFrame()

for df_name, df in dfs.items():
    result_df = result_df.join(df, how='outer')  # 'outer' joins all columns

print(result_df)

csv_file_path = 'DOW30_EOD_Data.csv'  # Specify the path where you want to save the CSV file
result_df.to_csv(csv_file_path)  # Set index=False to avoid writing row numbers

print(f'DataFrame has been saved to {csv_file_path}')

               AXP    AMGN      AAPL      BA     CAT   CSCO     CVX      GS  \
date                                                                          
2014-05-20   87.09  112.22  604.7104  129.58  101.56  24.12  122.50  156.35   
2014-05-21   87.86  113.49  606.3092  130.96  102.88  24.48  124.16  159.35   
2014-05-22   88.15  114.86  607.2696  131.84  103.08  24.38  123.63  159.85   
2014-05-23   88.78  114.33  614.1296  132.41  104.03  24.52  123.37  160.16   
2014-05-27   91.39  115.89  625.6292  134.17  103.68  24.71  122.84  161.77   
...            ...     ...       ...     ...     ...    ...     ...     ...   
2024-05-14  241.53  311.41  187.4300  180.76  358.18  48.95  163.61  458.47   
2024-05-15  241.70  319.04  189.7200  176.99  360.04  49.67  163.05  466.09   
2024-05-16  241.32  314.72  189.8400  182.96  350.72  48.34  161.09  464.52   
2024-05-17  242.82  312.47  189.8700  184.95  356.27  48.17  162.67  467.72   
2024-05-20  242.30  314.54  191.0400  186.61  362.75

# Finding Returns

In [ ]:
#now to find returns using pct_change()

ret = result_df.copy()

for i in range(1, len(symbols)+1):

  ret[symbols[i-1]] = ret[symbols[i-1]].pct_change()

  # 1. Drop NaN values
  ret = ret.dropna(subset=[symbols[i-1]])

  # 2. Convert to percentages
  ret[symbols[i-1]] = ret[symbols[i-1]] * 100

  # 3. Drop anything above 100%
  ret = ret[ret[symbols[i-1]] <= 100]

In [ ]:
ret

,AXP,AMGN,AAPL,BA,CAT,CSCO,CVX,GS,HD,HON,...,PG,TRV,UNH,CRM,VZ,V,WBA,WMT,DIS,DOW
date,,,,,,,,,,,,,,,,,,,,,
2019-03-21,0.947910,0.402888,3.683036,-0.919821,0.789414,1.276756,0.938327,0.133800,2.079527,1.382046,...,0.756980,2.314780,0.606786,2.103847,1.075082,1.333333,1.290014,0.425791,-1.209201,-1.646586
2019-03-22,-2.110535,-2.704675,-2.070839,-2.825329,-3.200060,-2.224694,-2.200858,-2.888272,-0.642207,-1.653518,...,-0.809835,0.394433,-1.956194,-3.264450,2.521873,-1.752246,-1.871069,-0.787402,-0.395730,-0.775827
2019-03-25,-0.383702,-0.058918,-1.209108,2.288980,1.240657,-0.018961,-0.162483,-0.238146,0.492715,0.064172,...,0.295101,0.037064,-0.093083,-0.383901,0.535475,-0.026132,-1.153661,-0.112027,-0.406542,1.131687
2019-03-26,0.421864,0.900370,-1.033167,-0.021595,0.350129,0.948227,1.009032,1.156437,-0.179249,0.994036,...,1.039623,0.022230,-1.405655,-0.919940,0.915446,1.483369,0.372832,0.152796,2.180165,-0.610376
2019-03-27,-0.474886,-1.035747,0.899406,1.034073,-0.485437,-0.169078,-1.079513,-0.346111,-0.047534,-0.234950,...,-0.116482,1.014965,-0.690250,-2.685069,0.412337,-0.695428,0.500646,-1.128968,0.127111,2.558854
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-05-14,1.206788,0.959637,0.617350,1.300157,0.420545,0.554643,-0.559169,1.082547,-0.134913,-0.289500,...,-0.066317,0.466340,0.418181,-0.259441,-0.123335,-0.590572,1.269316,-0.910445,-0.453558,0.488544
2024-05-15,0.070385,2.450146,1.221789,-2.085638,0.519292,1.470889,-0.342277,1.662050,2.399413,0.910388,...,0.452461,-1.489039,0.714175,3.880058,0.000000,1.353784,-2.506812,-0.050117,-2.448980,-0.989103
2024-05-16,-0.157220,-1.354062,0.063251,3.373072,-2.588601,-2.677673,-1.202085,-0.336845,-1.703617,0.760753,...,0.810762,2.271985,0.718771,-0.994644,-0.592739,-0.589698,2.515372,6.986462,0.583828,-0.609550


# Extracting as CSV

In [ ]:
csv_file_path = 'DOW30_Returns.csv'  # Specify the path where you want to save the CSV file
ret.to_csv(csv_file_path)  # Set index=False to avoid writing row numbers

print(f'DataFrame has been saved to {csv_file_path}')

DataFrame has been saved to DOW30_Returns.csv
